# Peyk logging-system showcase

A planned, five-scenario run through `data/`'s real sample documents, built specifically to
exercise — and make visible — the job/event history and per-stage artifact persistence added in
`docs-personal/central_logging_system.md`. Where [`demo.ipynb`](demo.ipynb) is the minimal
single-run walkthrough, this notebook is the "look what it can actually show you" tour: a table
view of the job/event database, plus sample crop/output images pulled straight out of each run's
persisted artifacts.

**Assumptions made in choosing documents/models below — override the `SCENARIOS` list if any of
these guessed wrong:**
- "CIB Arabic" → `data/cib_sample.pdf` (the project's flagship born-digital Arabic test doc —
  has text, tables, and a figure, so scenario 1 alone exercises every stage).
- "Low quality scanned" → `data/bdc_sample_true_scanned.pdf` (the one sample explicitly named as
  a true scan).
- "Regulatory document" → `` data/_الادوات المالية 1.pdf `` ("Financial Instruments" — an
  IFRS-style disclosure document, distinct from `cib_sample.pdf`).
- "Small quick gemini" → `gemini-3-1-flash-lite` (same tier `demo.py`/`demo.ipynb` already
  default to for figures) — reused everywhere a Gemini model is needed (figures, scenario 2c's
  table-full recognition, scenario 3's fullpage transcription) for cost/consistency. Swap in a
  stronger tier (e.g. `gemini-3-1-pro`) per scenario if quality matters more than cost for your
  run.

**The five scenarios:**

| # | Document | layout | tsr | ocr | cell_ocr | figures / fullpage | What it shows |
|---|---|---|---|---|---|---|---|
| 1 | CIB Arabic (born-digital) | heron | surya | paddleocr-vl | surya | gemini (figures) | Baseline: surya full-table recognition, paddleocr-vl whole-region OCR, heron layout, gemini figure captions — one job touching every stage. |
| 2a | Low-quality scan | heron | surya | paddleocr-vl | surya | gemini (figures) | Same config, `surya_smart_table_split` forced **off** — no correction pass on blurry/low-sharpness table crops. |
| 2b | Same scan | heron | surya | paddleocr-vl | surya | gemini (figures) | Same again, smart split **on** (the documented default) — direct duration/quality comparison against 2a. |
| 2c | Same scan | heron | **gemini** | paddleocr-vl | **gemini** | gemini (figures) | Tables routed through Gemini's full-table recognition instead of surya's — same document, a different backend entirely for the hardest stage. |
| 3 | Regulatory doc | — | — | — | — | **fullpage: gemini** | Bypasses the whole per-region pipeline (layout/tsr/ocr/figures never dispatch) for one whole-page-at-a-time Gemini transcription — the most different code path this SDK has. |

Requires Docker with GPU support, `peyk:dev` already built (see the repo README / `demo.py
--build-image`), and GCP credentials available (every scenario here calls Gemini for at least
one role) — same credential-loading cell as `demo.ipynb`. Run `uv sync --extra notebook` first
(pulls in `pandas`/`pillow` for the table/image views below, on top of `demo.ipynb`'s
jupyter/ipykernel).

In [ ]:
import shutil
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from PIL import Image

REPO_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT / "sdk" / "src"))

from peyk import Peyk, PipelineConfig, SmartSplitConfig, StageConfig

IMAGE = "peyk:dev"
INPUT_ROOT = REPO_ROOT / "hotstorage" / "showcase_input"
OUTPUT_ROOT = REPO_ROOT / "hotstorage" / "showcase_output"
CONFIG_ROOT = REPO_ROOT / "hotstorage" / "showcase_config"

peyk = Peyk(image=IMAGE)

## Credentials

Same as `demo.ipynb`: picked up automatically from `containers/peyk/.env`
(`AWS_BEARER_TOKEN_BEDROCK` — unused by this notebook's scenarios, but harmless to set) and
`containers/peyk/gcp-key.json` (`GOOGLE_APPLICATION_CREDENTIALS` — required; every scenario below
calls a Gemini model for at least one role).

In [ ]:
bedrock_env = REPO_ROOT / "containers" / "peyk" / ".env"
bedrock_token = None
if bedrock_env.exists():
    for line in bedrock_env.read_text().splitlines():
        if line.startswith("AWS_BEARER_TOKEN_BEDROCK="):
            bedrock_token = line.split("=", 1)[1].strip()

gcp_key = REPO_ROOT / "containers" / "peyk" / "gcp-key.json"
gcp_key = gcp_key if gcp_key.exists() else None
if gcp_key is None:
    print("WARNING: no containers/peyk/gcp-key.json found — every scenario below needs Gemini credentials.")

peyk.set_credentials(bedrock_bearer_token=bedrock_token, gcp_key_path=gcp_key)

## Scenario definitions

Each entry is one document + one `PipelineConfig`. `id` is used to namespace that scenario's own
input/output/config directories under `hotstorage/showcase_*` and, more importantly, its
persisted artifacts (`peyk.artifacts.root/<stage>/<job_id>/...`) so five runs never collide.

In [ ]:
GEMINI = "gemini-3-1-flash-lite"  # the "small quick gemini" — see the intro cell for why

SCENARIOS = [
    {
        "id": "1_cib_surya_table",
        "label": "1 — CIB Arabic: surya full-table + paddleocr-vl OCR + heron layout + Gemini figures",
        "doc": "cib_sample.pdf",
        "config": PipelineConfig(
            layout=StageConfig(model="heron"),
            tsr=StageConfig(model="surya"),
            ocr=StageConfig(model="paddleocr-vl", lang="arabic"),
            cell_ocr=StageConfig(model="surya"),
            figures=StageConfig(model=GEMINI),
        ),
    },
    {
        "id": "2a_scan_nosplit",
        "label": "2a — Low-quality scan: same config, smart table-split OFF",
        "doc": "bdc_sample_true_scanned.pdf",
        "config": PipelineConfig(
            layout=StageConfig(model="heron"),
            tsr=StageConfig(model="surya"),
            ocr=StageConfig(model="paddleocr-vl", lang="arabic"),
            cell_ocr=StageConfig(model="surya"),
            figures=StageConfig(model=GEMINI),
            surya_smart_table_split=SmartSplitConfig(enabled=False),
        ),
    },
    {
        "id": "2b_scan_split",
        "label": "2b — Same scan: smart table-split ON",
        "doc": "bdc_sample_true_scanned.pdf",
        "config": PipelineConfig(
            layout=StageConfig(model="heron"),
            tsr=StageConfig(model="surya"),
            ocr=StageConfig(model="paddleocr-vl", lang="arabic"),
            cell_ocr=StageConfig(model="surya"),
            figures=StageConfig(model=GEMINI),
            surya_smart_table_split=SmartSplitConfig(enabled=True),
        ),
    },
    {
        "id": "2c_scan_gemini_table",
        "label": "2c — Same scan: Gemini full-table recognition instead of surya",
        "doc": "bdc_sample_true_scanned.pdf",
        "config": PipelineConfig(
            layout=StageConfig(model="heron"),
            tsr=StageConfig(model=GEMINI),
            ocr=StageConfig(model="paddleocr-vl", lang="arabic"),
            cell_ocr=StageConfig(model=GEMINI),
            figures=StageConfig(model=GEMINI),
        ),
    },
    {
        "id": "3_regulatory_fullpage",
        "label": "3 — Regulatory document: fullpage transcription via Gemini",
        "doc": "_\u0627\u0644\u0627\u062f\u0648\u0627\u062a \u0627\u0644\u0645\u0627\u0644\u064a\u0629 1.pdf",
        "config": PipelineConfig(fullpage=StageConfig(model=GEMINI)),
    },
]

for scenario in SCENARIOS:
    scenario["config"].validate()
print(f"{len(SCENARIOS)} scenarios defined and validated.")

## Run every scenario

`run()` takes a whole *directory* of input PDFs, so each scenario gets its own single-document
input directory (`stage_single_doc`) rather than sharing `hotstorage/input`. Sidecars
(`peyk-vllm-surya`/`peyk-vllm-paddleocr`) are started at most once each — `ensure_ready` below
checks `is_ready()` first, so scenarios 2a/2b (which need the exact same two sidecars as
scenario 1) don't pay surya's ~14-20 minute cold start a second and third time. Scenario 2c only
needs `paddleocr` (its tables route to Gemini, not surya) and scenario 3 needs neither — both
skip straight to `run()`.

`persist_artifacts=True` on every call is what makes the sample-image section further down
possible; `stream_logs=True` so you can watch each run's real progress rather than staring at a
blank cell.

In [ ]:
def stage_single_doc(doc_filename: str, scenario_id: str) -> Path:
    scenario_input = INPUT_ROOT / scenario_id
    if scenario_input.exists():
        shutil.rmtree(scenario_input)
    scenario_input.mkdir(parents=True, exist_ok=True)
    src = REPO_ROOT / "data" / doc_filename
    shutil.copy(src, scenario_input / src.name)
    return scenario_input


running_sidecars: set[str] = set()


def ensure_ready(names: set[str]) -> None:
    to_start = [n for n in names if n not in running_sidecars]
    for name in to_start:
        print(f"  starting {name} sidecar...")
        peyk.sidecars.start(name)
    for name in names:
        peyk.sidecars.wait_ready(name)
        running_sidecars.add(name)
    if names and not to_start:
        print(f"  already running: {sorted(names)}")
    elif not names:
        print("  no sidecars needed for this config")

In [ ]:
results = []

for scenario in SCENARIOS:
    print(f"\n=== {scenario['label']} ===")
    input_dir = stage_single_doc(scenario["doc"], scenario["id"])
    output_dir = OUTPUT_ROOT / scenario["id"]
    peyk.configure(scenario["config"], config_dir=CONFIG_ROOT / scenario["id"])
    ensure_ready(scenario["config"].sidecar_requirements())
    result = peyk.run(input_dir=input_dir, output_dir=output_dir, persist_artifacts=True, stream_logs=True)
    print(f"\n--> exit_code={result.exit_code} job_id={result.job_id}")
    results.append({"scenario": scenario, "result": result})

print("\nAll scenarios finished.")

## Job table

A direct table view of `peyk.jobs` (the SQLite job history) — one row per scenario, regardless of
whether `persist_artifacts` was used.

In [ ]:
job_rows = []
for r in results:
    job = peyk.jobs.get_job(r["result"].job_id)
    duration_s = (job.ended_at - job.started_at) if job.ended_at and job.started_at else None
    job_rows.append({
        "scenario": r["scenario"]["label"],
        "job_id": job.job_id,
        "status": job.status,
        "exit_code": job.exit_code,
        "duration_s": round(duration_s, 1) if duration_s is not None else None,
    })

jobs_df = pd.DataFrame(job_rows)
jobs_df

## Event timeline

Every `dispatch_start`/`dispatch_end`/`stub`/`error` event across all five jobs, tagged by
scenario. This is the same data `peyk.jobs.get_events(job_id)` returns per job — concatenated
here so cross-scenario comparisons (surya vs. Gemini table recognition, split on vs. off) are one
`groupby` away instead of five separate lookups.

In [ ]:
event_rows = []
for r in results:
    for event in peyk.jobs.get_events(r["result"].job_id):
        event_rows.append({
            "scenario": r["scenario"]["label"],
            "stage": event.stage,
            "event": event.event,
            "model": event.model,
            "duration_s": event.duration_s,
            "exit_code": event.exit_code,
            "doc_stem": event.doc_stem,
            "region_id": event.region_id,
            "message": event.message,
            "artifact_path": event.artifact_path,
        })

events_df = pd.DataFrame(event_rows)
events_df

### The comparison this whole showcase is built around

`table_full`'s `dispatch_end` duration, side by side: scenario 1 vs. 2a vs. 2b isolates smart
table-split's effect on the same document/backend; 2b vs. 2c isolates surya vs. Gemini for the
exact same (split-enabled) run. Any `stub` rows here also flag which scenario, if any, fell back
to a placeholder for a table region.

In [ ]:
table_events = events_df[events_df["stage"].isin(["tsr", "table_full"])]
display(table_events[table_events["event"] == "dispatch_end"][["scenario", "stage", "model", "duration_s"]])

stub_events = events_df[events_df["event"] == "stub"]
if len(stub_events):
    print("\nStub fallbacks (missing/failed dispatch results):")
    display(stub_events[["scenario", "stage", "doc_stem", "region_id"]])
else:
    print("\nNo stub fallbacks in any scenario — every region got a real result.")

## Sample images from each stage

Only possible because every run above passed `persist_artifacts=True`. For each scenario, this
walks `peyk.artifacts.root/<stage>/<job_id>/` across every stage that actually dispatched, and
shows up to two images per stage — page renders, region/cell crops, TSR structure
visualizations, whatever that stage produced. Nothing here is a hardcoded filename: it's a plain
glob, so it stays correct regardless of which documents/configs the cells above end up using.

In [ ]:
def show_stage_samples(job_id: str, label: str, max_images: int = 2) -> None:
    root = peyk.artifacts.root
    print(f"\n--- {label} ---\njob {job_id}")
    if not root.exists():
        print("(no artifacts persisted yet)")
        return
    found_any = False
    for stage_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        job_dir = stage_dir / job_id
        if not job_dir.is_dir():
            continue
        images = sorted(job_dir.rglob("*.png"))[:max_images]
        if not images:
            continue
        found_any = True
        print(f"[{stage_dir.name}] {len(images)} sample(s):")
        for img_path in images:
            print(f"  {img_path.relative_to(root)}")
            display(Image.open(img_path))
    if not found_any:
        print("(no image artifacts found for this job)")


for r in results:
    show_stage_samples(r["result"].job_id, r["scenario"]["label"])

## Wrap-up

Five real jobs, five different `layout`/`tsr`/`ocr`/`cell_ocr`/`fullpage` combinations, all
traced through the same `peyk.jobs` history and the same stage-partitioned artifact tree — no
per-scenario logging code, no manual log-scraping to figure out which stage a stub or a slow
table came from. See `docs-personal/central_logging_system.md` for the design behind this, and
`sdk/README.md`'s "Job history & artifacts" section for the plain API.

Clean up afterward:

```python
# peyk.stop_sidecars()
# for r in results:
#     peyk.artifacts.cleanup(job_id=r["result"].job_id)
```

In [ ]:
# peyk.stop_sidecars()
# for r in results:
#     peyk.artifacts.cleanup(job_id=r["result"].job_id)